# In the paper ["Harsh", "Balanced", "Lenient"] == ["Bad", "Middle", "Good"] here.

## We perform prompt optimisation with:
## 1. Expert Annotations + Score from experts
## 2. Score-only from experts

## No_experts stands for score-only signal.

### Basically all annotated training bundles comes with an expert annotations and a score out of 5. No_experts stands for only using the score/5 to guide prompt optimisation which is misleading but I'm too lazy to change it.

In [8]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from google.colab import userdata
my_secret_key = userdata.get('API_KEY')


if my_secret_key:
  print("Token retrieved successfully.")
else:
  print("Token not found in Colab Secrets.")

from openai import OpenAI

client = OpenAI(
    # This is the default and can be omitted
    api_key = my_secret_key,
)


import asyncio
from openai import AsyncOpenAI

async_client = AsyncOpenAI(api_key = my_secret_key,
)  # make sure this is your actual key

Mounted at /content/drive
Token retrieved successfully.


In [9]:
import numpy as np
import csv
import pickle as pkl
import pickle
import copy
import re
import random
import matplotlib.pyplot as plt
import itertools
import json
import time
import os
import openai


!pip install scipy statsmodels
!pip install openpyxl

import pandas as pd
from scipy.stats import pearsonr, spearmanr
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

In [33]:
!rm -rf LLM4BEAR
!git clone --depth 1 --filter=blob:none --sparse https://github.com/anon5159753/LLM4BEAR.git
!cd LLM4BEAR && git sparse-checkout set "Bundle Annotations"
!cd LLM4BEAR && git sparse-checkout add 1_EGPO

import os

path = "/content/drive/MyDrive/EGPO/filter_results/"

try:
    os.makedirs(path, exist_ok=True)
    print(f"Successfully created: {path}")
except Exception as e:
    print(f"An error occurred: {e}")

Cloning into 'LLM4BEAR'...
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 15 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (15/15), 4.81 KiB | 4.81 MiB/s, done.
remote: Enumerating objects: 1, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 1 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (1/1), 67 bytes | 67.00 KiB/s, done.
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 9 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (9/9), 210.67 KiB | 14.04 MiB/s, done.
remote: Enumerating objects: 101, done.
remote: Counting objects: 100% (101/101), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 101 (delta 28), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 

# Part 1: Evaluate test annotated bundles using refined prompts

In [16]:
file_path = '/content/LLM4BEAR/Bundle Annotations/electronic_annotations.xlsx'

electronic_dataframe = pd.read_excel(file_path)

electronic_testing_scores = electronic_dataframe.iloc[72:137, 2].tolist()


file_path = '/content/LLM4BEAR/Bundle Annotations/clothing_annotations.xlsx'

clothing_dataframe = pd.read_excel(file_path)

clothing_testing_scores = clothing_dataframe.iloc[80:151 , 2].tolist()



file_path = '/content/LLM4BEAR/Bundle Annotations/food_annotations.xlsx'

food_dataframe = pd.read_excel(file_path)

food_testing_scores = food_dataframe.iloc[80:150 , 2].tolist()




electronic_info = pd.read_pickle("/content/LLM4BEAR/Bundle Annotations/electronic_bundle_info.pkl")


electronic_bundles = [
    {"bundle_intent": electronic_info[3][i],
     "items": [{"title": electronic_info[0][i][j],
                "image": "/content/LLM4BEAR/Bundle Annotations/electronics/" + str(electronic_info[2][i][j]) + ".jpg",
                "description": electronic_info[1][i][j],
                } for j in range(len(electronic_info[0][i]))]

     } for i in range(len(electronic_info[0]))
]


electronic_testing_bundles = [electronic_bundles[i] for i in range(72, len(electronic_bundles))]




clothing_info = pd.read_pickle("/content/LLM4BEAR/Bundle Annotations/clothing_bundle_info.pkl")


clothing_bundles = [
    {"bundle_intent": clothing_info[3][i],
     "items": [{"title": clothing_info[0][i][j],
                "image": "/content/LLM4BEAR/Bundle Annotations/clothing/" + str(clothing_info[2][i][j]) + ".jpg",
                "description": clothing_info[1][i][j],
                } for j in range(len(clothing_info[0][i]))]

     } for i in range(len(clothing_info[0]))
]


clothing_testing_bundles = [clothing_bundles[i] for i in range(80, len(clothing_bundles))]





food_info = pd.read_pickle("/content/LLM4BEAR/Bundle Annotations/food_bundle_info.pkl")


food_bundles = [
    {"bundle_intent": food_info[3][i],
     "items": [{"title": food_info[0][i][j],
                "image": "/content/LLM4BEAR/Bundle Annotations/food/" + str(food_info[2][i][j]) + ".jpg",
                "description": food_info[1][i][j],
                } for j in range(len(food_info[0][i]))]

     } for i in range(len(food_info[0]))
]


food_testing_bundles = [food_bundles[i] for i in range(80, 150)]


In [17]:
def input_strings(intent_list, item_list, description_list):
    l = len(intent_list)
    string_list = []
    for i in range(l):
        intent = intent_list[i]
        items = item_list[i]
        descriptions = description_list[i]
        item_str = "\n".join([f"{i + 1}. {title}" for i, title in enumerate(items)])

        string_list.append(f"Intent: {intent}\nBundle Items:\n{item_str}\n")
    return string_list

In [18]:
electronic_testing_strings = input_strings([bundle["bundle_intent"] for bundle in electronic_testing_bundles],
                                   [[item["title"] for item in bundle["items"]] for bundle in electronic_testing_bundles],
                                   [[item["description"] for item in bundle["items"]] for bundle in electronic_testing_bundles])

clothing_testing_strings = input_strings([bundle["bundle_intent"] for bundle in clothing_testing_bundles],
                                   [[item["title"] for item in bundle["items"]] for bundle in clothing_testing_bundles],
                                   [[item["description"] for item in bundle["items"]] for bundle in clothing_testing_bundles])

food_testing_strings = input_strings([bundle["bundle_intent"] for bundle in food_testing_bundles],
                                   [[item["title"] for item in bundle["items"]] for bundle in food_testing_bundles],
                                   [[item["description"] for item in bundle["items"]] for bundle in food_testing_bundles])

In [ ]:
async def single_request(user, system=None, seed_value=None):

    if system:
        message = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    else:
        message = [{"role": "user", "content": user}]

    # Reimplemented the robust retry loop with exponential backoff
    for delay_secs in (2**x for x in range(0, 3)):
        try:
            response = await async_client.chat.completions.create(
                model="gpt-4o-mini",
                messages=message,
                temperature=0,
                max_tokens=8000,
                seed=seed_value
            )
            return response.choices[0].message.content.strip()
        except openai.OpenAIError as e:
            randomness_collision_avoidance = random.randint(0, 1000) / 1000.0
            sleep_dur = delay_secs + randomness_collision_avoidance
            print(f"Error: {e}. Retrying in {round(sleep_dur, 2)} seconds.")
            await asyncio.sleep(sleep_dur)

    # Return None if all retries fail
    return None


async def openai_request(prompts, system=None, batch_size=128, delay=0):

    results = []

    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        tasks = [
            single_request(d["prompts"], system=system, seed_value=42)
            for j, d in enumerate(batch)
        ]

        batch_results = await asyncio.gather(*tasks)
        results.extend(batch_results)
        print(f"✅ Sending batch {i // batch_size + 1} — sleeping for {delay}s...\n")
        await asyncio.sleep(delay)

    return results


In [ ]:
def extract_json_simple_replace(response_text):
    """
    Extracts a JSON object from a string that has a "===JSON_START===" separator.

    This function isolates the JSON by finding the first '{' and last '}'
    to ensure it works correctly even with markdown fences or extra whitespace.

    Args:
        response_text (str): The full string containing the separator and JSON.

    Returns:
        dict: The parsed JSON object as a Python dictionary, or None if an error occurs.
    """
    try:
        # 1. Get the text after the separator
        json_part = response_text.split("===JSON_START===")[1]

        # 2. Find the boundaries of the JSON object
        first_brace = json_part.find('{')
        last_brace = json_part.rfind('}')

        # 3. Slice the string to get only the valid JSON
        # This will fail gracefully in the json.loads() if a brace isn't found
        json_string = json_part[first_brace : last_brace + 1]

        # 4. Parse the clean string
        parsed_json = json.loads(json_string)
        # print("I'M GOING")
        return parsed_json

    except IndexError:
        print("Error: The separator '===JSON_START===' was not found.")
        return None
    except json.JSONDecodeError:
        print("Error: Could not find or parse a valid JSON object after the separator.")
        return None


def extract_bundle_score(response):
    json_schema = extract_json_simple_replace(response)
    if json_schema is not None:
        try:
            given_score = float(json_schema['score'])
        except:
            given_score = None
    else:
        given_score = None

    return given_score

json_bad = "After you have completed your full written analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that summarizes your conclusions from Part 1. Do not include any other text after the separator.\n"\
            "**JSON Schema:**\n"\
            "```json\n"\
            "{{\n"\
            "is_poor_quality_bundle: str, yes/no response, do not provide anything other than yes or no.\n"\
            "is_acceptable_quality_bundle: str, yes/no response, do not provide anything other than yes or no.\n"\
            "score: float, bundle quality out of 5.\n"\
            "}}"


json_middle = "After you have completed your full written analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that summarizes your conclusions from Part 1. Do not include any other text after the separator.\n"\
            "**JSON Schema:**\n"\
            "```json\n"\
            "{{\n"\
            "needs_improvement_bundle: str, yes/no response, do not provide anything other than yes or no.\n"\
            "is_good_quality_bundle: str, yes/no response, do not provide anything other than yes or no.\n"\
            "score: float, bundle quality out of 5.\n"\
            "}}"

json_good = "After you have completed your full written analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that summarizes your conclusions from Part 1. Do not include any other text after the separator.\n"\
            "**JSON Schema:**\n"\
            "```json\n"\
            "{{\n"\
            "needs_improvement_bundle: str, yes/no response, do not provide anything other than yes or no.\n"\
            "is_high_quality_bundle: str, yes/no response, do not provide anything other than yes or no.\n"\
            "score: float, bundle quality out of 5.\n"\
            "}}"


def extract_bundle_verdict(response, consideration):
    json_schema = extract_json_simple_replace(response)
    if json_schema is not None:
        try:
            if consideration == "1-2":
                given_verdict_1 = json_schema['is_poor_quality_bundle']
                given_verdict_2 = json_schema['is_acceptable_quality_bundle']
                given_verdict = [given_verdict_1.lower(), given_verdict_2.lower()]
            elif consideration == "3":
                given_verdict_1 = json_schema['needs_improvement_bundle']
                given_verdict_2 = json_schema['is_good_quality_bundle']
                given_verdict = [given_verdict_1.lower(), given_verdict_2.lower()]
            elif consideration == "4-5":
                given_verdict_1 = json_schema['needs_improvement_bundle']
                given_verdict_2 = json_schema['is_high_quality_bundle']
                given_verdict = [given_verdict_1.lower(), given_verdict_2.lower()]
            else:
                given_verdict = None
        except:
            given_verdict = None
    else:
        given_verdict = None

    return given_verdict

def true_rmse(score_list, true_list):
    # Ensure lists are NumPy arrays to perform vectorized operations
    score_array = np.array(score_list, dtype=object)
    true_array = np.array(true_list, dtype=object)

    # Create a boolean mask to find all non-None scores
    valid_indices = score_array != None

    # Apply the mask to both arrays to get only the valid scores
    valid_scores = score_array[valid_indices].astype(float)
    valid_trues = true_array[valid_indices].astype(float)

    # Check if there are any valid scores to prevent ZeroDivisionError
    if len(valid_scores) == 0:
        return 0.0

    return np.sqrt(np.mean((valid_scores - valid_trues)**2))



def inverse_reward(collective_rmse, base_reward=1.0, epsilon=0.1):
    """
    Calculates reward from a collective error using an inverse function.
    'epsilon' prevents division by zero.
    """
    return base_reward / (collective_rmse + epsilon)

In [ ]:
async def separate_considerations(char, initial_prompt, sample_data, true_scores, json_addition, constant_metrics = "", consideration = ""):


    epsilon = 0.5

    reward = 0

    prompt_list = [{"prompts": data + "\n" + json_addition} for data in sample_data]

    responses = await openai_request(prompt_list, initial_prompt + constant_metrics)

    if consideration == "":
        scores = [extract_bundle_score(i) for i in responses]

        verdicts = None

    elif consideration == "1-2" or consideration == "3" or consideration == "4-5":
        verdicts = [extract_bundle_verdict(i, consideration) for i in responses]

        scores = [extract_bundle_score(i) for i in responses]



    target_scores = true_scores

    fuck_ups = 0

    if consideration == "1-2":
        for i in range(len(responses)):

            try:

                if target_scores[i] > 2 and verdicts[i][0] == "no" and verdicts[i][1] == "yes":
                    reward += 1
                elif target_scores[i] < 3 and verdicts[i][0] == "yes" and verdicts[i][1] == "no":
                    reward += 1
            except:
                print("response error")

    elif consideration == "4-5":
        for i in range(len(responses)):

            try:

                if target_scores[i] > 3 and verdicts[i][1] == "yes" and verdicts[i][0] == "no":
                    reward += 1
                elif target_scores[i] < 4 and verdicts[i][1] == "no" and verdicts[i][0] == "yes":
                    reward += 1

            except:
                print("response error")


    elif consideration == "3":

        for i in range(len(responses)):
            try:

                if target_scores[i] < 4 and verdicts[i][0] == "yes" and verdicts[i][1] == "no":
                    reward += 1
                elif target_scores[i] > 3 and verdicts[i][0] == "no" and verdicts[i][1] == "yes":
                    reward += 1

            except:
                print("response error")

    elif consideration == "":

        collective_rmse = true_rmse(scores, target_scores)

        reward = inverse_reward(collective_rmse, base_reward = 3, epsilon = 0.5)



    filename = f"/content/drive/MyDrive/EGPO/filter_results/{char}.pkl"

    with open(filename, 'wb') as f:
        pickle.dump([reward, responses, verdicts, scores], f)

    return

In [19]:
dataset_string = ["electronic_dataset", "clothing_dataset", "food_dataset"]
datasets = [electronic_testing_strings, clothing_testing_strings, food_testing_strings]
dataset_scores = [electronic_testing_scores, clothing_testing_scores, food_testing_scores]

adding_metrics = "\nFunctionality Integration: Describe how a user would utilize this collection of items to achieve their primary goal. Considering the entire workflow, is this a complete and logical set of items for the task, or is there an irrelevant or missing item? \n"\
              "Similarity: What is the common theme or category that connects these items?\n"\
              "Complementarity: Are these items more valuable together than they would be if sold separately? Does the presence of one item create a clear reason to buy the other(s)?\n"\
              "Diversity: Does the variety of items in this bundle cater to a broad set of related needs for a single user, or does the mix of items seem unfocused and random?\n"


json_bad = "After you have completed your full written analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that summarizes your conclusions from Part 1. Do not include any other text after the separator.\n"\
            "**JSON Schema:**\n"\
            "```json\n"\
            "{{\n"\
            "score: float, bundle quality out of 5.\n"\
            "is_poor_quality_bundle: str, yes/no response, do not provide anything other than yes or no.\n"\
            "is_acceptable_quality_bundle: str, yes/no response, do not provide anything other than yes or no.\n"\
            "}}"


json_middle = "After you have completed your full written analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that summarizes your conclusions from Part 1. Do not include any other text after the separator.\n"\
            "**JSON Schema:**\n"\
            "```json\n"\
            "{{\n"\
            "score: float, bundle quality out of 5.\n"\
            "needs_improvement_bundle: str, yes/no response, do not provide anything other than yes or no.\n"\
            "is_good_quality_bundle: str, yes/no response, do not provide anything other than yes or no.\n"\
            "}}"

json_good = "After you have completed your full written analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that summarizes your conclusions from Part 1. Do not include any other text after the separator.\n"\
            "**JSON Schema:**\n"\
            "```json\n"\
            "{{\n"\
            "score: float, bundle quality out of 5.\n"\
            "needs_improvement_bundle: str, yes/no response, do not provide anything other than yes or no.\n"\
            "is_high_quality_bundle: str, yes/no response, do not provide anything other than yes or no.\n"\
            "}}"

In [20]:
# initial_prompt = "You are an expert bundle strategist tasked distinguishing a $quality$/5 quality bundle. Your response should be a single judgement, is this bundle a $quality$/5 bundle?: yes/no. \n"\
#                   "Your task is to diligently complete subtasks that can help guide your analysis step by step:\n" \
#                   "1. Based on the stated intent, come up with how combinations of items can interact to fulfill the intent.\n" \
#                   "2. If from your understanding, the combinations do not meet the stated intent, develop a new intent that your combinations can fulfill.\n" \
#                   "3. Based on your reasoning and analysis, you are to make a yes/no judgement on the bundle. $criteria$" \
#                   "4. Now, you are to analyse the bundle: \n"

# Importance graph prompts are better

initial_importance_prompt = "You are an expert bundle strategist tasked distinguishing a $quality$/5 quality bundle. Your response should be a single judgement, is this bundle a $quality$/5 bundle?: yes/no. \n"\
                  "Your task is to diligently complete subtasks that can help guide your analysis step by step:\n" \
                  "1. Based on the stated intent, come up with how combinations of items can interact to fulfill the intent.\n" \
                  "2. If from your understanding, the combinations do not meet the stated intent, develop a new intent that your combinations can fulfill.\n" \
                  "3. Design a importance graph analysis for each bundle in the format:\n"\
                  "a. [Most Important Bundle Item] — [Role: Primary/Secondary/Tertiary]\n"\
                  "Reason: [Your explanation for this item's rank in this specific scenario]\n"\
                  "b. [2nd Most Important Bundle Item] — [Role: Primary/Secondary/Tertiary]\n"\
                  "Reason: [Your explanation]\n"\
                  "(...and so on for all other items)\n"\
                  "4. Based on your reasoning and analysis, you are to make two mutually exclusive yes/no judgements on the bundle. $criteria$" \
                  "5. Now, you are to analyse the bundle: \n"

bad_bundle_criteria = "\nFollow this criteria when evaluating the bundle:\n"\
                "1-2 - Poor: Some items or no items are connected thematically, but more than one modification needs to be made to guarantee the acceptability of the bundle.\n"

middle_bundle_criteria = "\nFollow this criteria when evaluating the bundle:\n"\
                "1-3 - Needs Improvement: One or more modifications needs to be made in order to guarantee the acceptability of the bundle.\n"

good_bundle_criteria = "\nFollow this criteria when evaluating the bundle:\n"\
                "4-5 - Acceptable: No modifications need to be performed to be accepted by the end user.\n"


# initial_prompt_bad_bundle = initial_prompt.replace("$criteria$",
#                                                          bad_bundle_criteria).replace("$quality$", "1-2")

# initial_prompt_middle_bundle = initial_prompt.replace("$criteria$",
#                                                          middle_bundle_criteria).replace("$quality$", "1-3")

# initial_prompt_good_bundle = initial_prompt.replace("$criteria$",
#                                                          good_bundle_criteria).replace("$quality$", "4-5")


initial_importance_prompt_bad_bundle = initial_importance_prompt.replace("$criteria$",
                                                         bad_bundle_criteria).replace("$quality$", "1-2")

initial_importance_prompt_middle_bundle = initial_importance_prompt.replace("$criteria$",
                                                         middle_bundle_criteria).replace("$quality$", "1-3")

initial_importance_prompt_good_bundle = initial_importance_prompt.replace("$criteria$",
                                                         good_bundle_criteria).replace("$quality$", "4-5")


considerations = ["1-2", "3", "4-5"]

jsons = [json_bad, json_middle, json_good]


# Refined Prompts are supplied for you already, though if you have refined your own, from EGPO.ipynb, you can take it from your Google Drive

In [28]:
initial_importance_charizards = ["initial_bad_importance_prompt", "initial_middle_importance_prompt", "initial_good_importance_prompt",
                                 ]
initial_importance_prompts_to_test = [initial_importance_prompt_bad_bundle, initial_importance_prompt_middle_bundle, initial_importance_prompt_good_bundle,
                                      ]



electronic_importance_charizards = ["bad_evaluator_electronic_importance", "middle_evaluator_electronic_importance", "good_evaluator_electronic_importance",
                                    "bad_evaluator_electronic_importance_no_experts", "middle_evaluator_electronic_importance_no_experts", "good_evaluator_electronic_importance_no_experts"]



electronic_importance_prompts_to_test = []

for char in electronic_importance_charizards:
    # filename = f"/content/drive/MyDrive/EGPO/final_prompts/Refined_{char}.pkl"
    filename = f"/content/LLM4BEAR/1_EGPO/final_prompts/Refined_{char}.pkl"

    with open(filename, 'rb') as f:
        top_3_prompts, _ = pickle.load(f)

    electronic_importance_prompts_to_test.append(top_3_prompts)


clothing_importance_charizards = ["bad_evaluator_clothing_importance", "middle_evaluator_clothing_importance", "good_evaluator_clothing_importance",
                                    "bad_evaluator_clothing_importance_no_experts", "middle_evaluator_clothing_importance_no_experts", "good_evaluator_clothing_importance_no_experts"]


clothing_importance_prompts_to_test = []

for char in clothing_importance_charizards:
    # filename = f"/content/drive/MyDrive/EGPO/final_prompts/Refined_{char}.pkl"
    filename = f"/content/LLM4BEAR/1_EGPO/final_prompts/Refined_{char}.pkl"

    with open(filename, 'rb') as f:
        top_3_prompts, _ = pickle.load(f)

    clothing_importance_prompts_to_test.append(top_3_prompts)



food_importance_charizards = ["bad_evaluator_food_importance", "middle_evaluator_food_importance", "good_evaluator_food_importance",
                                    "bad_evaluator_food_importance_no_experts", "middle_evaluator_food_importance_no_experts", "good_evaluator_food_importance_no_experts"]


food_importance_prompts_to_test = []

for char in food_importance_charizards:
    # filename = f"/content/drive/MyDrive/EGPO/final_prompts/Refined_{char}.pkl"
    filename = f"/content/LLM4BEAR/1_EGPO/final_prompts/Refined_{char}.pkl"

    with open(filename, 'rb') as f:
        top_3_prompts, _ = pickle.load(f)

    food_importance_prompts_to_test.append(top_3_prompts)


In [ ]:
# j in range(3) because 3 top prompts
# k in range(0,1) because electronic refined prompts should evaluate electronic bundles only
# basically, k is a parameter if you want to use an electronic prompt to evaluate a different domain i.e., clothing or food.

In [29]:
from google.colab import files
import pickle

def download_locally(char):
    # Path to your file
    filename = f"/content/drive/MyDrive/PO4ISR/filter_results/{char}.pkl"

    # This triggers a direct browser download to your 'Downloads' folder
    files.download(filename)

# Just call the function with the character name
# download_locally("your_char_name")

In [30]:

electronic_importance_charizards = ["bad_evaluator_electronic_importance", "middle_evaluator_electronic_importance", "good_evaluator_electronic_importance",
                                    "bad_evaluator_electronic_importance_no_experts", "middle_evaluator_electronic_importance_no_experts", "good_evaluator_electronic_importance_no_experts"]

clothing_importance_charizards = ["bad_evaluator_clothing_importance", "middle_evaluator_clothing_importance", "good_evaluator_clothing_importance",
                                    "bad_evaluator_clothing_importance_no_experts", "middle_evaluator_clothing_importance_no_experts", "good_evaluator_clothing_importance_no_experts"]

food_importance_charizards = ["bad_evaluator_food_importance", "middle_evaluator_food_importance", "good_evaluator_food_importance",
                                    "bad_evaluator_food_importance_no_experts", "middle_evaluator_food_importance_no_experts", "good_evaluator_food_importance_no_experts"]



# for char in electronic_importance_charizards:
#     download_locally(char)
# for char in clothing_importance_charizards:
#     download_locally(char)
# for char in food_importance_charizards:
#     download_locally(char)

for i in range(len(initial_importance_charizards)):
    for k in range(3):

        char=dataset_string[k] + "_" + initial_importance_charizards[i]

        download_locally(char)



for i in range(len(electronic_importance_charizards)):
    for j in range(3):

        for k in range(0,1):


            char=dataset_string[k] + "_" + electronic_importance_charizards[i] + "_" + str(j)

            download_locally(char)


for i in range(len(clothing_importance_charizards)):
    for j in range(3):

        for k in range(1,2):


            char=dataset_string[k] + "_" + clothing_importance_charizards[i] + "_" + str(j)

            download_locally(char)


for i in range(len(food_importance_charizards)):
    for j in range(3):

        for k in range(2,3):


            char=dataset_string[k] + "_" + food_importance_charizards[i] + "_" + str(j)

            download_locally(char)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
for i in range(len(initial_importance_charizards)):
    for k in range(3):

        await separate_considerations(char=dataset_string[k] + "_" + initial_importance_charizards[i],
                                      initial_prompt=initial_importance_prompts_to_test[i],
                                      sample_data=datasets[k],
                                      true_scores=dataset_scores[k],
                                      json_addition=jsons[int(i%3)],
                                      constant_metrics = adding_metrics,
                                      consideration = considerations[int(i%3)])



for i in range(len(electronic_importance_charizards)):
    for j in range(3):

        for k in range(0,1):


            await separate_considerations(char=dataset_string[k] + "_" + electronic_importance_charizards[i] + "_" + str(j),
                                          initial_prompt=electronic_importance_prompts_to_test[i][j],
                                          sample_data=datasets[k],
                                          true_scores=dataset_scores[k],
                                          json_addition=jsons[int(i%3)],
                                          constant_metrics = adding_metrics,
                                          consideration = considerations[int(i%3)])



for i in range(len(clothing_importance_charizards)):
    for j in range(3):

        for k in range(1,2):


            await separate_considerations(char=dataset_string[k] + "_" + clothing_importance_charizards[i] + "_" + str(j),
                                          initial_prompt=clothing_importance_prompts_to_test[i][j],
                                          sample_data=datasets[k],
                                          true_scores=dataset_scores[k],
                                          json_addition=jsons[int(i%3)],
                                          constant_metrics = adding_metrics,
                                          consideration = considerations[int(i%3)])


for i in range(len(food_importance_charizards)):
    for j in range(3):

        for k in range(2,3):


            await separate_considerations(char=dataset_string[k] + "_" + food_importance_charizards[i] + "_" + str(j),
                                          initial_prompt=food_importance_prompts_to_test[i][j],
                                          sample_data=datasets[k],
                                          true_scores=dataset_scores[k],
                                          json_addition=jsons[int(i%3)],
                                          constant_metrics = adding_metrics,
                                          consideration = considerations[int(i%3)])



# Part 1: Determine which prompts are best

In [32]:
def purge_none_pairs(list_a, list_b):
    """
    Takes two lists and returns new lists containing only the pairs where
    neither entry is None.

    Args:
        list_a (list): Scores from the first rater.
        list_b (list): Scores from the second rater.

    Returns:
        tuple: (purged_list_a, purged_list_b)
    """
    if len(list_a) != len(list_b):
        raise ValueError("Input lists must have the same length.")

    purged_a = []
    purged_b = []

    # Iterate through the lists simultaneously using zip
    for score_a, score_b in zip(list_a, list_b):
        # Check if BOTH entries are not None
        if score_a is not None and score_b is not None:
            purged_a.append(score_a)
            purged_b.append(score_b)

    return purged_a, purged_b

def calculate_reliability_metrics_complete(scores_rater_a_raw, scores_rater_b_raw):
    """
    1. Purges None entries from both lists.
    2. Calculates ICC, Pearson's r, and Spearman's rho on the cleaned data.
    """

    # 1. PURGE STEP
    scores_rater_a, scores_rater_b = purge_none_pairs(scores_rater_a_raw, scores_rater_b_raw)

    if len(scores_rater_a) < 3: # Need at least 3 items to run a stable ANOVA
        return {'Error': 'Not enough valid paired data points (min 3 required for stability).'}

    n_items = len(scores_rater_a)

    # 2. Prepare Data for ICC (Long Format)
    data = pd.DataFrame({
        'Item': [f'Item_{i}' for i in range(n_items)] * 2,
        'Rater': ['A'] * n_items + ['B'] * n_items,
        'Score': scores_rater_a + scores_rater_b
    })

    # 3. Perform Two-Way ANOVA to get Mean Squares (MS)
    model = ols('Score ~ C(Item) + C(Rater)', data=data).fit()
    anova_table = anova_lm(model, typ=1)

    # Extract Mean Squares (MS)
    try:
        MS_I = anova_table.loc['C(Item)', 'mean_sq']  # MS_Items
        MS_R = anova_table.loc['C(Rater)', 'mean_sq']  # MS_Raters
        MS_E = anova_table.loc['Residual', 'mean_sq'] # MS_Error
    except KeyError:
        return {'Error': 'ANOVA Mean Squares extraction failed. Data structure might be perfectly uniform.'}

    # 4. Calculate ICC (Absolute Agreement, Single Rater: ICC(A,1))
    k = 2  # Number of raters (A and B)
    n = n_items

    denominator = MS_I + (k - 1) * MS_E + (k / n) * (MS_R - MS_E)

    if denominator != 0:
        icc_value = (MS_I - MS_E) / denominator
    else:
        icc_value = float('nan')

    # 5. Calculate Pearson's r (Linear Correlation)
    pearson_corr, _ = pearsonr(scores_rater_a, scores_rater_b)

    # 6. Calculate Spearman's rho (Rank Correlation)
    spearman_rho, _ = spearmanr(scores_rater_a, scores_rater_b)

    # 7. Compile and Return Results
    return {
        'N_Valid_Pairs': n_items,
        'ICC_Absolute_Agreement': icc_value,
        'Pearson_r': pearson_corr,
        'Spearman_rho': spearman_rho
    }




In [31]:
def the_extractler(char):

    # you can use your own, or just use the results provided.

    # filename = f"/content/drive/MyDrive/EGPO/filter_results/{char}.pkl"
    filename = f"/content/LLM4BEAR/1_EGPO/filter_results/{char}.pkl"


    with open(filename, 'rb') as f:
        reward, responses, verdicts, scores = pickle.load(f)

    return scores

In [34]:
initial_importance_charizards = ["initial_bad_importance_prompt", "initial_middle_importance_prompt", "initial_good_importance_prompt"]

electronic_importance_charizards = ["bad_evaluator_electronic_importance", "middle_evaluator_electronic_importance", "good_evaluator_electronic_importance",
                                    "bad_evaluator_electronic_importance_no_experts", "middle_evaluator_electronic_importance_no_experts", "good_evaluator_electronic_importance_no_experts"]

clothing_importance_charizards = ["bad_evaluator_clothing_importance", "middle_evaluator_clothing_importance", "good_evaluator_clothing_importance",
                                    "bad_evaluator_clothing_importance_no_experts", "middle_evaluator_clothing_importance_no_experts", "good_evaluator_clothing_importance_no_experts"]

food_importance_charizards = ["bad_evaluator_food_importance", "middle_evaluator_food_importance", "good_evaluator_food_importance",
                                    "bad_evaluator_food_importance_no_experts", "middle_evaluator_food_importance_no_experts", "good_evaluator_food_importance_no_experts"]


dataset_string = ["electronic_dataset", "clothing_dataset", "food_dataset"]

In [35]:
initial_elec_bad_score = the_extractler(dataset_string[0] + "_" + initial_importance_charizards[0])
initial_elec_middle_score = the_extractler(dataset_string[0] + "_" + initial_importance_charizards[1])
initial_elec_good_score = the_extractler(dataset_string[0] + "_" + initial_importance_charizards[2])

initial_clo_bad_scores = the_extractler(dataset_string[1] + "_" + initial_importance_charizards[0])
initial_clo_middle_scores = the_extractler(dataset_string[1] + "_" + initial_importance_charizards[1])
initial_clo_good_scores = the_extractler(dataset_string[1] + "_" + initial_importance_charizards[2])

initial_food_bad_scores = the_extractler(dataset_string[2] + "_" + initial_importance_charizards[0])
initial_food_middle_scores = the_extractler(dataset_string[2] + "_" + initial_importance_charizards[1])
initial_food_good_scores = the_extractler(dataset_string[2] + "_" + initial_importance_charizards[2])


In [36]:
refined_elec_bad_score_0 = the_extractler(dataset_string[0] + "_" + electronic_importance_charizards[0] + "_" + str(0))
refined_elec_bad_score_1 = the_extractler(dataset_string[0] + "_" + electronic_importance_charizards[0] + "_" + str(1))
refined_elec_bad_score_2 = the_extractler(dataset_string[0] + "_" + electronic_importance_charizards[0] + "_" + str(2))

refined_elec_middle_score_0 = the_extractler(dataset_string[0] + "_" + electronic_importance_charizards[1] + "_" + str(0))
refined_elec_middle_score_1 = the_extractler(dataset_string[0] + "_" + electronic_importance_charizards[1] + "_" + str(1))
refined_elec_middle_score_2 = the_extractler(dataset_string[0] + "_" + electronic_importance_charizards[1] + "_" + str(2))

refined_elec_good_score_0 = the_extractler(dataset_string[0] + "_" + electronic_importance_charizards[2] + "_" + str(0))
refined_elec_good_score_1 = the_extractler(dataset_string[0] + "_" + electronic_importance_charizards[2] + "_" + str(1))
refined_elec_good_score_2 = the_extractler(dataset_string[0] + "_" + electronic_importance_charizards[2] + "_" + str(2))


refined_elec_bad_score_no_experts_0 = the_extractler(dataset_string[0] + "_" + electronic_importance_charizards[3] + "_" + str(0))
refined_elec_bad_score_no_experts_1 = the_extractler(dataset_string[0] + "_" + electronic_importance_charizards[3] + "_" + str(1))
refined_elec_bad_score_no_experts_2 = the_extractler(dataset_string[0] + "_" + electronic_importance_charizards[3] + "_" + str(2))

refined_elec_middle_score_no_experts_0 = the_extractler(dataset_string[0] + "_" + electronic_importance_charizards[4] + "_" + str(0))
refined_elec_middle_score_no_experts_1 = the_extractler(dataset_string[0] + "_" + electronic_importance_charizards[4] + "_" + str(1))
refined_elec_middle_score_no_experts_2 = the_extractler(dataset_string[0] + "_" + electronic_importance_charizards[4] + "_" + str(2))

refined_elec_good_score_no_experts_0 = the_extractler(dataset_string[0] + "_" + electronic_importance_charizards[5] + "_" + str(0))
refined_elec_good_score_no_experts_1 = the_extractler(dataset_string[0] + "_" + electronic_importance_charizards[5] + "_" + str(1))
refined_elec_good_score_no_experts_2 = the_extractler(dataset_string[0] + "_" + electronic_importance_charizards[5] + "_" + str(2))

In [37]:
refined_clo_bad_score_0 = the_extractler(dataset_string[1] + "_" + clothing_importance_charizards[0] + "_" + str(0))
refined_clo_bad_score_1 = the_extractler(dataset_string[1] + "_" + clothing_importance_charizards[0] + "_" + str(1))
refined_clo_bad_score_2 = the_extractler(dataset_string[1] + "_" + clothing_importance_charizards[0] + "_" + str(2))

refined_clo_middle_score_0 = the_extractler(dataset_string[1] + "_" + clothing_importance_charizards[1] + "_" + str(0))
refined_clo_middle_score_1 = the_extractler(dataset_string[1] + "_" + clothing_importance_charizards[1] + "_" + str(1))
refined_clo_middle_score_2 = the_extractler(dataset_string[1] + "_" + clothing_importance_charizards[1] + "_" + str(2))

refined_clo_good_score_0 = the_extractler(dataset_string[1] + "_" + clothing_importance_charizards[2] + "_" + str(0))
refined_clo_good_score_1 = the_extractler(dataset_string[1] + "_" + clothing_importance_charizards[2] + "_" + str(1))
refined_clo_good_score_2 = the_extractler(dataset_string[1] + "_" + clothing_importance_charizards[2] + "_" + str(2))


refined_clo_bad_score_no_experts_0 = the_extractler(dataset_string[1] + "_" + clothing_importance_charizards[3] + "_" + str(0))
refined_clo_bad_score_no_experts_1 = the_extractler(dataset_string[1] + "_" + clothing_importance_charizards[3] + "_" + str(1))
refined_clo_bad_score_no_experts_2 = the_extractler(dataset_string[1] + "_" + clothing_importance_charizards[3] + "_" + str(2))

refined_clo_middle_score_no_experts_0 = the_extractler(dataset_string[1] + "_" + clothing_importance_charizards[4] + "_" + str(0))
refined_clo_middle_score_no_experts_1 = the_extractler(dataset_string[1] + "_" + clothing_importance_charizards[4] + "_" + str(1))
refined_clo_middle_score_no_experts_2 = the_extractler(dataset_string[1] + "_" + clothing_importance_charizards[4] + "_" + str(2))

refined_clo_good_score_no_experts_0 = the_extractler(dataset_string[1] + "_" + clothing_importance_charizards[5] + "_" + str(0))
refined_clo_good_score_no_experts_1 = the_extractler(dataset_string[1] + "_" + clothing_importance_charizards[5] + "_" + str(1))
refined_clo_good_score_no_experts_2 = the_extractler(dataset_string[1] + "_" + clothing_importance_charizards[5] + "_" + str(2))

In [38]:
refined_food_bad_score_0 = the_extractler(dataset_string[2] + "_" + food_importance_charizards[0] + "_" + str(0))
refined_food_bad_score_1 = the_extractler(dataset_string[2] + "_" + food_importance_charizards[0] + "_" + str(1))
refined_food_bad_score_2 = the_extractler(dataset_string[2] + "_" + food_importance_charizards[0] + "_" + str(2))

refined_food_middle_score_0 = the_extractler(dataset_string[2] + "_" + food_importance_charizards[1] + "_" + str(0))
refined_food_middle_score_1 = the_extractler(dataset_string[2] + "_" + food_importance_charizards[1] + "_" + str(1))
refined_food_middle_score_2 = the_extractler(dataset_string[2] + "_" + food_importance_charizards[1] + "_" + str(2))

refined_food_good_score_0 = the_extractler(dataset_string[2] + "_" + food_importance_charizards[2] + "_" + str(0))
refined_food_good_score_1 = the_extractler(dataset_string[2] + "_" + food_importance_charizards[2] + "_" + str(1))
refined_food_good_score_2 = the_extractler(dataset_string[2] + "_" + food_importance_charizards[2] + "_" + str(2))


refined_food_bad_score_no_experts_0 = the_extractler(dataset_string[2] + "_" + food_importance_charizards[3] + "_" + str(0))
refined_food_bad_score_no_experts_1 = the_extractler(dataset_string[2] + "_" + food_importance_charizards[3] + "_" + str(1))
refined_food_bad_score_no_experts_2 = the_extractler(dataset_string[2] + "_" + food_importance_charizards[3] + "_" + str(2))

refined_food_middle_score_no_experts_0 = the_extractler(dataset_string[2] + "_" + food_importance_charizards[4] + "_" + str(0))
refined_food_middle_score_no_experts_1 = the_extractler(dataset_string[2] + "_" + food_importance_charizards[4] + "_" + str(1))
refined_food_middle_score_no_experts_2 = the_extractler(dataset_string[2] + "_" + food_importance_charizards[4] + "_" + str(2))

refined_food_good_score_no_experts_0 = the_extractler(dataset_string[2] + "_" + food_importance_charizards[5] + "_" + str(0))
refined_food_good_score_no_experts_1 = the_extractler(dataset_string[2] + "_" + food_importance_charizards[5] + "_" + str(1))
refined_food_good_score_no_experts_2 = the_extractler(dataset_string[2] + "_" + food_importance_charizards[5] + "_" + str(2))

In [39]:
def analyze_all_scores(initial_scores, refined_scores, refined_no_exp_scores, testing_scores):

    # Structure the inputs logically
    domains = ['elec', 'clo', 'food']
    personalities = ['bad', 'middle', 'good']
    prompts = [0, 1, 2]

    # Unpack expert scores
    expert_map = dict(zip(domains, testing_scores))

    all_results = {}

    # 1. Loop through ALL 63 INDIVIDUAL COMPARISONS (3x3x7)
    for dom_idx, domain in enumerate(domains):

        # We need the scores for each personality
        # Example: initial_scores[0] is initial_elec_scores (which is [bad, middle, good])

        # Access the initial scores for the current domain: [bad, middle, good]
        initial_domain_scores = initial_scores[dom_idx]

        # Access the refined scores for the current domain: [P0_bad, P1_bad, P2_bad, P0_middle, ..., P2_good]
        refined_domain_w_exp = refined_scores[dom_idx]
        refined_domain_no_exp = refined_no_exp_scores[dom_idx]

        for p_idx, personality in enumerate(personalities):

            # --- BASELINE COMPARISON ---
            base_llm_scores = initial_domain_scores[p_idx]
            key = f'{domain}_{personality}_baseline'
            all_results[key] = calculate_reliability_metrics_complete(base_llm_scores, expert_map[domain])

            # --- 6 REFINEMENT COMPARISONS (P0, P1, P2 * W/EXP, NO_EXP) ---

            # The indices for refined lists are tricky: bad=0-2, middle=3-5, good=6-8
            start_index = p_idx * len(prompts) # 0 for bad, 3 for middle, 6 for good

            for p_num in prompts:
                current_idx = start_index + p_num

                # 1. Refined W/ Expert
                refined_w_exp_llm = refined_domain_w_exp[current_idx]
                key_w_exp = f'{domain}_{personality}_P{p_num}_w_exp'
                all_results[key_w_exp] = calculate_reliability_metrics_complete(refined_w_exp_llm, expert_map[domain])

                # 2. Refined W/O Expert
                refined_no_exp_llm = refined_domain_no_exp[current_idx]
                key_no_exp = f'{domain}_{personality}_P{p_num}_no_exp'
                all_results[key_no_exp] = calculate_reliability_metrics_complete(refined_no_exp_llm, expert_map[domain])

    return all_results


def avg_lists(big_list):
    all_guys = []
    for small_list in big_list:
        avg_guys = []
        for i in range(len(small_list[0])):
            avg_guys.append((small_list[0][i] + small_list[1][i] + small_list[2][i])/3)

        all_guys.append(avg_guys)
    return all_guys

In [40]:
# --- STEP 1: Define the LLM input lists based on the domains ---

# 1. INITIAL SCORES (Structure: [ [elec_bad, elec_middle, elec_good], [clo_bad, clo_middle, clo_good], [food_bad, food_middle, food_good] ])
initial_scores = [
    [initial_elec_bad_score, initial_elec_middle_score, initial_elec_good_score],
    [initial_clo_bad_scores, initial_clo_middle_scores, initial_clo_good_scores],
    [initial_food_bad_scores, initial_food_middle_scores, initial_food_good_scores]
]

unrefined_avg = avg_lists(initial_scores)


best_refined_scores = [[refined_elec_bad_score_2, refined_elec_middle_score_2, refined_elec_good_score_2],
                       [refined_clo_bad_score_1, refined_clo_middle_score_0, refined_clo_good_score_0],
                       [refined_food_bad_score_1, refined_food_middle_score_0, refined_food_good_score_1]
                       ]

best_refined_avg = avg_lists(best_refined_scores)

best_refined_no_exp_scores = [[refined_elec_bad_score_no_experts_1, refined_elec_middle_score_no_experts_1, refined_elec_good_score_no_experts_2],
                              [refined_clo_bad_score_no_experts_2, refined_clo_middle_score_no_experts_2, refined_clo_good_score_no_experts_0],
                              [refined_food_bad_score_no_experts_0, refined_food_middle_score_no_experts_2, refined_food_good_score_no_experts_0]
                              ]

best_refined_no_exp_avg = avg_lists(best_refined_no_exp_scores)

# 2. REFINED SCORES (Structure: [ [elec_P0_bad, elec_P1_bad, ..., elec_P2_good], [clo_P0_bad, ...], [food_P0_bad, ...] ])
# NOTE: The refined lists MUST be a single list containing all 9 prompt/personality combinations for that domain.
refined_scores = [
    [
        refined_elec_bad_score_0, refined_elec_bad_score_1, refined_elec_bad_score_2,
        refined_elec_middle_score_0, refined_elec_middle_score_1, refined_elec_middle_score_2,
        refined_elec_good_score_0, refined_elec_good_score_1, refined_elec_good_score_2
    ],
    [
        refined_clo_bad_score_0, refined_clo_bad_score_1, refined_clo_bad_score_2,
        refined_clo_middle_score_0, refined_clo_middle_score_1, refined_clo_middle_score_2,
        refined_clo_good_score_0, refined_clo_good_score_1, refined_clo_good_score_2
    ],
    [
        refined_food_bad_score_0, refined_food_bad_score_1, refined_food_bad_score_2,
        refined_food_middle_score_0, refined_food_middle_score_1, refined_food_middle_score_2,
        refined_food_good_score_0, refined_food_good_score_1, refined_food_good_score_2
    ]
]

# 3. REFINED SCORES NO EXPERTS (Same structure as above)
refined_no_exp_scores = [
    [
        refined_elec_bad_score_no_experts_0, refined_elec_bad_score_no_experts_1, refined_elec_bad_score_no_experts_2,
        refined_elec_middle_score_no_experts_0, refined_elec_middle_score_no_experts_1, refined_elec_middle_score_no_experts_2,
        refined_elec_good_score_no_experts_0, refined_elec_good_score_no_experts_1, refined_elec_good_score_no_experts_2
    ],
    [
        refined_clo_bad_score_no_experts_0, refined_clo_bad_score_no_experts_1, refined_clo_bad_score_no_experts_2,
        refined_clo_middle_score_no_experts_0, refined_clo_middle_score_no_experts_1, refined_clo_middle_score_no_experts_2,
        refined_clo_good_score_no_experts_0, refined_clo_good_score_no_experts_1, refined_clo_good_score_no_experts_2
    ],
    [
        refined_food_bad_score_no_experts_0, refined_food_bad_score_no_experts_1, refined_food_bad_score_no_experts_2,
        refined_food_middle_score_no_experts_0, refined_food_middle_score_no_experts_1, refined_food_middle_score_no_experts_2,
        refined_food_good_score_no_experts_0, refined_food_good_score_no_experts_1, refined_food_good_score_no_experts_2
    ]
]

# 4. TESTING SCORES (Already structured correctly)
testing_scores = [electronic_testing_scores, clothing_testing_scores, food_testing_scores]


# --- STEP 2: Call the analysis function ---

final_results = analyze_all_scores(
    initial_scores,
    refined_scores,
    refined_no_exp_scores,
    testing_scores
)


In [41]:
import pandas as pd
import json

def format_results_for_display(results_dict):
    """
    Converts the nested results dictionary into a clean, comparative Pandas DataFrame.
    """

    # List to hold the rows of the DataFrame
    data_rows = []

    # Iterate through each comparison result
    for key, metrics in results_dict.items():
        if 'Error' in metrics:
            row = {'Comparison': key, 'Status': metrics['Error']}
        else:
            # Parse the key to extract components
            parts = key.split('_')

            # This logic assumes the naming convention from analyze_all_scores_v3
            if 'initial' in key:
                domain, personality, stage = parts[0], parts[1], 'Baseline'
                prompt = 'N/A'
                condition = 'N/A'
            else:
                domain = parts[0]
                personality = parts[1]
                stage = 'Refined'
                prompt = parts[2].replace('P', '') # P0, P1, P2 -> 0, 1, 2
                condition = 'W/Expert' if 'w_exp' in key else 'No Expert'

            row = {
                'Domain': domain.capitalize(),
                'Personality': personality.capitalize(),
                'Stage': stage,
                'Prompt_ID': prompt,
                'Condition': condition,
                'N_Pairs': metrics.get('N_Valid_Pairs', 'N/A'),
                'ICC_Absolute_Agreement': metrics.get('ICC_Absolute_Agreement', float('nan')),
                'Pearson_r': metrics.get('Pearson_r', float('nan')),
                'Spearman_rho': metrics.get('Spearman_rho', float('nan'))
            }
        data_rows.append(row)

    df = pd.DataFrame(data_rows)

    # --- Sorting and Formatting for Optimal Viewing ---

    # Sort primarily by Domain, then by Personality, then by Stage/Prompt ID
    df = df.sort_values(by=['Domain', 'Personality', 'Stage', 'Prompt_ID'], ascending=[True, True, False, True])

    # Format the critical metrics to 3 decimal places for clean comparison
    float_cols = ['ICC_Absolute_Agreement', 'Pearson_r', 'Spearman_rho']
    for col in float_cols:
        if col in df.columns:
            df[col] = df[col].map('{:.3f}'.format)

    # Highlight the key metric (ICC) to make the best prompts stand out
    # Note: Requires a rendering environment like Jupyter/Colab/HTML, otherwise just prints text.

    return df

# --- How to Run and Print the Table ---

# Assuming 'final_results' holds the output from analyze_all_scores

formatted_df = format_results_for_display(final_results)

# 1. Print the entire table
print("\n--- Comprehensive LLM Reliability Comparison ---")
print(formatted_df.to_string())

# 2. Print filtered tables for selection (MOST USEFUL)
print("\n--- TOP ICC Selection (Best Prompt Choice) ---")
# This groups by the LLM personality and finds the highest ICC score for each.
for personality in ['bad', 'middle', 'good']:
    subset = formatted_df[formatted_df['Personality'] == personality.capitalize()]

    # Find the highest ICC score within this group (requires converting the string back to float for sorting)
    subset['ICC_float'] = pd.to_numeric(subset['ICC_Absolute_Agreement'], errors='coerce')

    print(f"\nPersonality: {personality.upper()} LLM")
    print(subset.sort_values(by='ICC_float', ascending=False).head(4).drop(columns='ICC_float').to_string(index=False))

    # Drop the temporary column for next loop
    del subset['ICC_float']





--- Comprehensive LLM Reliability Comparison ---
   Domain Personality    Stage Prompt_ID  Condition  N_Pairs ICC_Absolute_Agreement Pearson_r Spearman_rho
22    Clo         Bad  Refined         0   W/Expert       71                  0.311     0.541        0.601
23    Clo         Bad  Refined         0  No Expert       71                  0.313     0.406        0.390
24    Clo         Bad  Refined         1   W/Expert       71                  0.404     0.547        0.511
25    Clo         Bad  Refined         1  No Expert       71                  0.298     0.421        0.384
26    Clo         Bad  Refined         2   W/Expert       71                  0.300     0.467        0.495
27    Clo         Bad  Refined         2  No Expert       71                  0.402     0.512        0.499
21    Clo         Bad  Refined  baseline  No Expert       71                  0.160     0.430        0.390
36    Clo        Good  Refined         0   W/Expert       71                  0.537     0.539 

/tmp/ipython-input-3598813543.py:80: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  subset['ICC_float'] = pd.to_numeric(subset['ICC_Absolute_Agreement'], errors='coerce')
/tmp/ipython-input-3598813543.py:80: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  subset['ICC_float'] = pd.to_numeric(subset['ICC_Absolute_Agreement'], errors='coerce')
/tmp/ipython-input-3598813543.py:80: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = v

In [42]:
def export_results_to_excel(df: pd.DataFrame, filename: str = 'LLM_Reliability_Analysis.xlsx'):
    """
    Exports the formatted DataFrame to a downloadable Excel file.

    Args:
        df (pd.DataFrame): The DataFrame containing the formatted reliability results.
        filename (str): The name of the Excel file to create.
    """
    try:
        # Use the to_excel method with the openpyxl engine
        df.to_excel(filename, index=False, engine='openpyxl')
        print(f"\n✅ Success: Data exported to '{filename}'")
        print(f"You can now download this file from your current working directory.")
    except Exception as e:
        print(f"\n❌ Error during Excel export: {e}")
        print("Please ensure 'openpyxl' is installed (pip install openpyxl).")

# --- Example of How to Run ---

# Assume 'formatted_df' is the output from format_results_for_display(final_results)

# Call the function to create the Excel file:
export_results_to_excel(formatted_df, 'ICC_Comparison_Results.xlsx')


✅ Success: Data exported to 'ICC_Comparison_Results.xlsx'
You can now download this file from your current working directory.


In [43]:
def final_proxy_validation_icc(unrefined_avg ,best_refined_avg, best_refined_no_exp_avg, testing_scores):
    """
    Performs the final ICC comparison of the pooled LLM proxy scores against the Expert.
    """

    domains = ['elec', 'clo', 'food']

    # Structure the final comparisons to run
    comparisons = [
        # 0. Unrefined Avg
        ('proxy_unrefined_elec', unrefined_avg[0], testing_scores[0]),
        ('proxy_unrefined_clo', unrefined_avg[1], testing_scores[1]),
        ('proxy_unrefined_food', unrefined_avg[2], testing_scores[2]),

        # 1. W/ Expert Trained Proxies
        ('proxy_refined_w_exp_elec', best_refined_avg[0], testing_scores[0]),
        ('proxy_refined_w_exp_clo', best_refined_avg[1], testing_scores[1]),
        ('proxy_refined_w_exp_food', best_refined_avg[2], testing_scores[2]),

        # 2. W/O Expert Trained Proxies (for comparison)
        ('proxy_refined_no_exp_elec', best_refined_no_exp_avg[0], testing_scores[0]),
        ('proxy_refined_no_exp_clo', best_refined_no_exp_avg[1], testing_scores[1]),
        ('proxy_refined_no_exp_food', best_refined_no_exp_avg[2], testing_scores[2]),
    ]

    final_proxy_results = {}

    print("\n--- Running Final 6 Pooled Proxy ICC Validations ---")
    for key, llm_proxy_scores, expert_scores in comparisons:
        # We use the existing reliability calculation engine
        results = calculate_reliability_metrics_complete(llm_proxy_scores, expert_scores)
        final_proxy_results[key] = results
        print(f"  - Completed: {key}")

    return final_proxy_results


In [44]:
# --- EXECUTION ---
# You would call this function after defining your score lists:

final_proxy_results = final_proxy_validation_icc(
    unrefined_avg,
    best_refined_avg,
    best_refined_no_exp_avg,
    testing_scores
)

# # Then, display the results or export to Excel:
formatted_proxy_df = format_results_for_display(final_proxy_results)
print("\n========== FINAL POOLED PROXY VALIDATION RESULTS ==========")
print(formatted_proxy_df.to_string())
export_results_to_excel(formatted_proxy_df, 'Final_Proxy_Validation.xlsx')


--- Running Final 6 Pooled Proxy ICC Validations ---
  - Completed: proxy_unrefined_elec
  - Completed: proxy_unrefined_clo
  - Completed: proxy_unrefined_food
  - Completed: proxy_refined_w_exp_elec
  - Completed: proxy_refined_w_exp_clo
  - Completed: proxy_refined_w_exp_food
  - Completed: proxy_refined_no_exp_elec
  - Completed: proxy_refined_no_exp_clo
  - Completed: proxy_refined_no_exp_food

========== FINAL POOLED PROXY VALIDATION RESULTS ==========
  Domain Personality    Stage Prompt_ID  Condition  N_Pairs ICC_Absolute_Agreement Pearson_r Spearman_rho
6  Proxy     Refined  Refined        no  No Expert       65                  0.746     0.759        0.701
7  Proxy     Refined  Refined        no  No Expert       71                  0.570     0.594        0.611
8  Proxy     Refined  Refined        no  No Expert       70                  0.676     0.687        0.659
3  Proxy     Refined  Refined         w   W/Expert       65                  0.764     0.771        0.736
4  Prox

# It isn't well organised here, but the organised excel spreadsheet of the performance of Refined Prompts are in LLM4BEAR/1_EGPO/ICC_Pearson_Spearman_of_Best_Refined_Prompts.xlsx